# AI Engineer Challenge - Dot Indonesia
## Object Tracking Mobil dengan YOLO + Teori AI

**Kandidat:** Gymnastiar Al K

---
Notebook ini berisi implementasi object tracking menggunakan YOLO (Bagian 1A)
serta jawaban untuk soal teori (Bagian 2).

---
# Bagian 1A: Object Tracking Mobil dengan Pre-trained Model YOLO

Pipeline:
1. Install dependencies
2. Load pre-trained YOLO model (YOLOv8)
3. Download sample video
4. Deteksi + tracking kendaraan per frame
5. Visualisasi hasil

In [ ]:
# Install dependencies
# ultralytics untuk YOLO, opencv untuk video processing
!pip install -q ultralytics opencv-python-headless matplotlib numpy tqdm
print("dependencies installed")

In [ ]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from IPython.display import display, Video
from tqdm.notebook import tqdm
import os
from pathlib import Path

print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

### Load Pre-trained YOLO

Menggunakan YOLOv8s (small) — versi yang lebih akurat dari nano.
Dipilih karena CCTV aerial view memiliki objek kendaraan yang relatif kecil,
sehingga model yang lebih besar dibutuhkan untuk deteksi yang lebih baik.
Model sudah pre-trained di dataset COCO yang memiliki class kendaraan seperti car, bus, truck, motorcycle.

In [ ]:
model = YOLO('yolov8s.pt')  # small — lebih akurat dari nano untuk objek kecil

# Kelas kendaraan di COCO: 2=car, 3=motorcycle, 5=bus, 7=truck
vehicle_classes = [2, 3, 5, 7]

print(f"Model: yolov8s.pt (small)")
print(f"Target classes: {[model.names[c] for c in vehicle_classes]}")

### Download Sample Video

Menggunakan sample video dari sumber publik. Disarankan upload video sendiri
atau gunakan link video lalu lintas dari repositori open source.

In [ ]:
# Download sample traffic video
# Video dari antonmilev/TrafficDetection — crossroads traffic, 1920x1080, 25fps
VIDEO_URL = "https://raw.githubusercontent.com/antonmilev/TrafficDetection/main/vid.mp4"
VIDEO_PATH = 'traffic_crossroad.mp4'

!wget -q --show-progress -O $VIDEO_PATH $VIDEO_URL

if os.path.exists(VIDEO_PATH) and os.path.getsize(VIDEO_PATH) > 100000:
    print(f"Video downloaded: {os.path.getsize(VIDEO_PATH)/1024/1024:.1f} MB")
    cap_check = cv2.VideoCapture(VIDEO_PATH)
    print(f"Resolution: {int(cap_check.get(3))}x{int(cap_check.get(4))}, {int(cap_check.get(cv2.CAP_PROP_FPS))} fps")
    cap_check.release()
else:
    print("Download gagal. Upload video sendiri:")
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        VIDEO_PATH = list(uploaded.keys())[0]
        print(f"Using: {VIDEO_PATH}")

### Deteksi dan Tracking

Fungsi ini melakukan deteksi kendaraan menggunakan YOLO + ByteTrack.
ByteTrack mempertahankan ID objek antar frame sehingga tracking bersifat konsisten.

In [ ]:
def detect_and_track(frame, model, conf_threshold=0.3):
    """Deteksi kendaraan dan tracking dengan ByteTrack."""
    results = model.track(frame, persist=True, conf=conf_threshold, iou=0.5,
                         imgsz=1280, tracker='bytetrack.yaml', verbose=False)

    detections = []
    frame_out = frame.copy()

    if results[0].boxes is None:
        return frame_out, detections

    boxes = results[0].boxes.xyxy.cpu().numpy()
    confs = results[0].boxes.conf.cpu().numpy()
    cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
    track_ids = results[0].boxes.id.cpu().numpy().astype(int) if results[0].boxes.id is not None else [-1]*len(boxes)

    for box, conf, cls_id, tid in zip(boxes, confs, cls_ids, track_ids):
        if cls_id not in vehicle_classes:
            continue

        x1, y1, x2, y2 = map(int, box)
        label = f"{'ID:'+str(tid)+' ' if tid > 0 else ''}{model.names[cls_id]} {conf:.2f}"

        detections.append({
            'bbox': (x1, y1, x2, y2),
            'conf': float(conf),
            'class': model.names[cls_id],
            'track_id': int(tid)
        })

        # Warna berbeda per track ID
        color = (int(tid) * 127 % 255, int(tid) * 63 % 255, int(tid) * 255 % 255) if tid > 0 else (0, 255, 0)
        cv2.rectangle(frame_out, (x1, y1), (x2, y2), color, 2)

        # Label background
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
        cv2.rectangle(frame_out, (x1, y1-20), (x1+tw+5, y1), color, -1)
        cv2.putText(frame_out, label, (x1+2, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    return frame_out, detections

print("Function ready")

### Proses Video

Iterasi setiap frame, jalankan deteksi + tracking, lalu simpan sebagai video baru.

In [ ]:
def process_video(video_path, model, output_name='output_tracked.mp4', conf=0.3, max_frames=300):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        total = min(total, max_frames)

    out = cv2.VideoWriter(output_name, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    vehicle_counts = []

    print(f"Processing {total} frames ({fps} fps, {w}x{h})...")
    for i in tqdm(range(total)):
        ret, frame = cap.read()
        if not ret:
            break

        annotated, dets = detect_and_track(frame, model, conf)
        cv2.putText(annotated, f"Frame {i+1}/{total} | Vehicles: {len(dets)}",
                   (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
        out.write(annotated)
        vehicle_counts.append(len(dets))

    cap.release()
    out.release()

    print(f"\nDone. Output: {output_name}")
    print(f"Total frames processed: {len(vehicle_counts)}")
    print(f"Average vehicles/frame: {np.mean(vehicle_counts):.1f}")
    print(f"Max vehicles in a frame: {np.max(vehicle_counts)}")
    return output_name, vehicle_counts


out_video, counts = process_video(VIDEO_PATH, model, 'output_tracked.mp4', 0.3, 300)
print(f"\n>>> Variable 'counts'长度: {len(counts)}")

In [ ]:
# Install ffmpeg kalo belum ada
import subprocess, shutil
if not shutil.which('ffmpeg'):
    !apt-get install -qq ffmpeg > /dev/null 2>&1
    print('ffmpeg installed')

# Compress biar bisa ditampilin di Colab
COMPRESSED = 'output_compressed.mp4'
!ffmpeg -y -i output_tracked.mp4 -vcodec libx264 -crf 23 -preset fast $COMPRESSED -loglevel error

# Tampilkan video hasil
from IPython.display import Video, HTML
from base64 import b64encode

if os.path.exists(COMPRESSED) and os.path.getsize(COMPRESSED) > 1000:
    print(f"\n✅ Video result ({os.path.getsize(COMPRESSED)/1024:.0f} KB)")
    display(Video(COMPRESSED, width=800))
elif os.path.exists('output_tracked.mp4'):
    # Fallback: baca sebagai HTML5 video
    with open('output_tracked.mp4', 'rb') as f:
        video_data = b64encode(f.read()).decode()
    video_html = f'''<video width="800" controls>
        <source src="data:video/mp4;base64,{video_data}" type="video/mp4">
    </video>'''
    display(HTML(video_html))
else:
    print("\n⚠️ Video output tidak ditemukan. Cek proses video sebelumnya.")

### Sample Frame Results

Visualisasi beberapa frame untuk melihat detail deteksi.

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
total_fr = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
# Ambil 5 sample frame (grid 1x5 biar gak ada slot kosong)
indices = np.linspace(0, min(total_fr-1, 300), 5, dtype=int)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('Detection Results - Sample Frames', fontsize=14)

for ax, idx in zip(axes, indices):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
    ret, frame = cap.read()
    if not ret:
        continue
    annotated, dets = detect_and_track(frame, model)
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Frame {idx} | {len(dets)} vehicles", fontsize=10)
    ax.axis('off')

cap.release()
plt.tight_layout()
plt.show()

### Tracking Summary

In [ ]:
# Validasi data
if 'counts' not in dir() or len(counts) == 0:
    print("ERROR: Variable 'counts' kosong! Jalankan cell process_video dulu.")
else:
    print(f"Total frames processed: {len(counts)}")
    print(f"Total vehicle detections: {np.sum(counts)}")
    print(f"Average per frame: {np.mean(counts):.1f}")
    print(f"Max vehicles in a single frame: {np.max(counts)}")
    print(f"Min vehicles in a single frame: {np.min(counts)}")
    print(f"Model: yolov8s.pt | Tracker: ByteTrack | Confidence threshold: 0.3 | Inference size: 1280")

    # Distribution plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Plot 1: Line chart
    axes[0].plot(counts, alpha=0.7, linewidth=1, color='steelblue')
    axes[0].axhline(np.mean(counts), color='r', linestyle='--', label=f'Mean: {np.mean(counts):.1f}')
    axes[0].set_xlabel('Frame')
    axes[0].set_ylabel('Vehicle Count')
    axes[0].legend()
    axes[0].set_title('Vehicle Count per Frame')
    axes[0].grid(True, alpha=0.3)

    # Plot 2: Histogram
    axes[1].hist(counts, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    axes[1].axvline(np.mean(counts), color='r', linestyle='--', label=f'Mean: {np.mean(counts):.1f}')
    axes[1].set_xlabel('Vehicle Count')
    axes[1].set_ylabel('Frequency')
    axes[1].legend()
    axes[1].set_title('Distribution of Vehicle Counts')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

---
# Bagian 2: Teori Artificial Intelligence
---

### Soal 2a: Apa yang dimaksud dengan Artificial Intelligence (AI)?

Artificial Intelligence (AI) adalah cabang ilmu komputer yang berfokus pada pengembangan sistem yang mampu melakukan tugas yang biasanya membutuhkan kecerdasan manusia. AI bekerja dengan memproses data, mengenali pola, dan membuat keputusan atau prediksi berdasarkan pola tersebut.

**Dua contoh AI dalam kehidupan sehari-hari:**

1. **Sistem rekomendasi (Netflix, YouTube, Spotify)** — AI menganalisis riwayat interaksi pengguna untuk merekomendasikan konten yang relevan, menggunakan teknik collaborative filtering dan deep learning.

2. **Asisten virtual (Siri, Google Assistant, ChatGPT)** — Menggunakan Natural Language Processing (NLP) untuk memahami perintah suara atau teks, lalu memberikan respons yang sesuai.

---
*Referensi: Russell, S., & Norvig, P. (2020). Artificial Intelligence: A Modern Approach (4th ed.). Pearson.*

### Soal 2b: Perbedaan Supervised Learning dan Unsupervised Learning

| Aspek | Supervised Learning | Unsupervised Learning |
|-------|--------------------|-----------------------|
| Data | Berlabel (labeled) | Tidak berlabel (unlabeled) |
| Tujuan | Memprediksi output dari input | Menemukan struktur/kelompok dalam data |
| Feedback | Ada (dari ground truth) | Tidak ada |
| Contoh algoritma | Linear Regression, Random Forest, SVM | K-Means, DBSCAN, PCA |

**Contoh Supervised Learning:** Klasifikasi email spam/not-spam. Model dilatih dengan ribuan email yang sudah dilabeli, sehingga bisa memprediksi email baru masuk ke kategori mana.

**Contoh Unsupervised Learning:** Segmentasi pelanggan berdasarkan pola pembelian. Model mengelompokkan pelanggan ke dalam cluster tanpa label sebelumnya — misalnya cluster "pembeli reguler", "pembeli musiman", dll.

### Soal 3a: Apa itu Feature dalam Machine Learning?

Feature adalah variabel atau atribut yang dapat diukur dari data dan digunakan sebagai input model machine learning. Feature membantu model memahami pola dalam data untuk membuat prediksi.

**Contoh:** Untuk model prediksi harga rumah, feature dapat berupa luas tanah, jumlah kamar, lokasi, tahun dibangun, dan jarak ke pusat kota.

**Mengapa pemilihan feature penting?**

1. **Relevansi** — Feature yang tidak relevan menambah noise dan menurunkan akurasi.
2. **Curse of Dimensionality** — Semakin banyak feature, semakin banyak data yang dibutuhkan. Feature yang tidak informatif hanya memperbesar dimensi tanpa kontribusi.
3. **Interpretability** — Model dengan feature yang sedikit dan bermakna lebih mudah dijelaskan.
4. **Efisiensi komputasi** — Feature yang tepat membuat training lebih cepat.
5. **Overfitting** — Feature berlebihan dapat menyebabkan model menghafal noise.

Teknik seleksi feature yang umum: feature importance, PCA, mutual information, dan Recursive Feature Elimination (RFE).

---
*Referensi:*
- *Bellman, R. (1957). Dynamic Programming. Princeton University Press.*
- *Guyon, I., & Elisseeff, A. (2003). An introduction to variable and feature selection. Journal of Machine Learning Research, 3, 1157–1182.*

### Soal 3b: Apa itu Fine-tuning dalam Machine Learning?

Fine-tuning adalah teknik transfer learning di mana model yang sudah pre-trained pada dataset besar dan umum dilatih ulang secara spesifik pada dataset yang lebih kecil untuk tugas tertentu.

Prosesnya:
- Ambil model pre-trained (misal: YOLO, ResNet, BERT)
- Ganti layer akhir (head) sesuai tugas baru
- Latih ulang dengan learning rate kecil (cukup beberapa epoch karena model sudah memiliki pengetahuan dasar)

**Contoh kasus: Deteksi objek khusus — deteksi sampah pesisir**

YOLO yang pre-trained di COCO (80 kelas umum) dapat di-fine-tune untuk mendeteksi sampah plastik di wilayah pesisir — kelas yang tidak ada di COCO. Karena model sudah memahami konsep dasar objek (tepi, tekstur, bentuk), fine-tuning hanya membutuhkan ratusan gambar berlabel, bukan ribuan. Ini jauh lebih efisien dibandingkan training dari scratch.

**Manfaat fine-tuning:**
- Konvergensi lebih cepat (epoch lebih sedikit)
- Membutuhkan data lebih sedikit
- Biaya komputasi lebih rendah
- Performa lebih baik pada dataset kecil dibandingkan training dari nol

---
*Referensi:*
- *Yosinski, J., Clune, J., Bengio, Y., & Lipson, H. (2014). How transferable are features in deep neural networks? NeurIPS, 27.*
- *Pan, S. J., & Yang, Q. (2010). A survey on transfer learning. IEEE TKDE, 22(10), 1345–1359.*